# Pipeline — preprocessing + augmentation

**Étapes :**
1. Preprocessing des audios réels → `STG_RESPIRATORY_SOUNDS` (processed)
2. Génération des candidats augmentés (classes sous-représentées) à partir des audios brut (RAW)
3. Preprocessing des candidats augmentés (même pipeline)
4. Sélection qualité par similarité MFCC cosinus
5. Upload des meilleurs augmentés → `STG_RESPIRATORY_SOUNDS_AUGMENTED`
6. Création de la table d'entraînement finale (réels + augmentés sélectionnés)

In [ ]:
import os, sys, zipfile, io

import_dir = sys._xoptions.get("snowflake_import_directory")
zip_name = "libroza.zip"
final_lib_dir = "/tmp/site-packages"

os.makedirs(final_lib_dir, exist_ok=True)

if import_dir:
    zip_path = os.path.join(import_dir, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as outer:
        for name in outer.namelist():
            if name.endswith(".whl"):
                # Read wheel into memory, never touch disk
                whl_bytes = outer.read(name)
                with zipfile.ZipFile(io.BytesIO(whl_bytes), 'r') as whl:
                    whl.extractall(final_lib_dir)

    if final_lib_dir not in sys.path:
        sys.path.insert(0, final_lib_dir)

import librosa
print(f"librosa {librosa.__version__} OK")

In [ ]:
import numpy as np
import pandas as pd
import soundfile as sf
import tempfile, random, warnings
from scipy.signal import butter, filtfilt
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# ── Stages ────────────────────────────────────────────────────────────────────
SOURCE_STAGE      = "@M2_ISD_EQUIPE_1_DB.RAW.STG_RESPIRATORY_SOUNDS"
PROCESSED_STAGE   = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS"
AUGMENTED_STAGE   = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS_AUG"
FEATURE_STAGE     = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES"
AUG_FEATURE_STAGE = "@M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES_AUG"

# ── Tables ────────────────────────────────────────────────────────────────────
SOURCE_METADATA_TABLE  = "M2_ISD_EQUIPE_1_DB.RAW.RESPIRATORY_SOUNDS_METADATA"
PROC_METADATA_TABLE    = "M2_ISD_EQUIPE_1_DB.PROCESSED.RESPIRATORY_SOUNDS_METADATA"
AUG_METADATA_TABLE     = "M2_ISD_EQUIPE_1_DB.PROCESSED.RESPIRATORY_SOUNDS_AUG_METADATA"
FEATURE_METADATA_TABLE = "M2_ISD_EQUIPE_1_DB.PROCESSED.RESPIRATORY_FEATURES_METADATA"
AUG_FEATURE_META_TABLE = "M2_ISD_EQUIPE_1_DB.PROCESSED.RESPIRATORY_FEATURES_AUG_METADATA"
TRAINING_VIEW          = "M2_ISD_EQUIPE_1_DB.PROCESSED.TRAINING_DATA_V"

# ── Audio params ──────────────────────────────────────────────────────────────
TARGET_SR          = 22050
MIN_DURATION_S     = 4
TARGET_DURATION_S  = 6
SILENCE_DB        = 30
BANDPASS_LOW       = 100
BANDPASS_HIGH      = 2000

# ── Augmentation params ───────────────────────────────────────────────────────
MAX_AUG_RATIO      = 0.30   # max 30% d'augmentés dans la classe finale
GENERATION_FACTOR  = 3      # générer 3× le besoin → sélection qualité
N_MFCC_SCORE       = 13     # coefficients pour le scoring cosinus
RANDOM_SEED        = 42

# ── Mapping classe → sous-dossier stage ──────────────────────────────────────
CLASS_TO_DIR = {
    "Asthma":    "asthma",
    "Bronchial": "bronchial",
    "COPD":      "copd",
    "Pneumonia":  "pneumonia",
    "Healthy":   "healthy",
}

# ── Création des stages si besoin ────────────────────────────────────────────
for stage_fqn in [
    "M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS",
    "M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_SOUNDS_AUG",
    "M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES",
    "M2_ISD_EQUIPE_1_DB.PROCESSED.STG_RESPIRATORY_FEATURES_AUG",
]:
    session.sql(f"CREATE STAGE IF NOT EXISTS {stage_fqn}").collect()

print("Configuration OK")

In [ ]:
# ── Filtre passe-bande ────────────────────────────────────────────────────────
def bandpass_filter(y, sr, low=BANDPASS_LOW, high=BANDPASS_HIGH, order=4):
    """
    Butterworth passe-bande 100-2000 Hz.
    Guard : filtfilt nécessite au moins padlen+1 samples (≈ 3*order*2 = 24).
    En dessous on retourne le signal brut plutôt que de crasher.
    """
    min_samples = 3 * order * 2 + 1
    if len(y) < min_samples:
        return y  # signal trop court pour filtrer proprement
    nyq = sr / 2
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, y)


# ── Trim silence (début ET fin) ───────────────────────────────────────────────
def trim_silence(y, sr, top_db=SILENCE_DB):
    """
    Supprime le silence en début ET en fin.
    Retourne (y_trimmed, leading_s, trailing_s).

    Fix v2 : l'ancienne version ne supprimait que le début
    (audio[trim_index[0]:] ignorait trim_index[1]).
    """
    _, indices = librosa.effects.trim(y, top_db=top_db)
    y_trimmed    = y[indices[0]:indices[1]]
    leading_s    = indices[0] / sr
    trailing_s   = (len(y) - indices[1]) / sr
    return y_trimmed, leading_s, trailing_s


# ── Pad / crop ────────────────────────────────────────────────────────────────
def pad_or_crop(y, sr, target_s=TARGET_DURATION_S):
    """Padding à droite ou crop au début pour atteindre target_s secondes."""
    target = int(target_s * sr)
    if len(y) < target:
        return np.pad(y, (0, target - len(y)))
    return y[:target]


# ── Normalisation amplitude ───────────────────────────────────────────────────
def normalize_amplitude(y, target_peak=0.95):
    """
    Ramène le pic d'amplitude à target_peak.
    Fix v2 : absent de v1, causait des disparités de volume entre fichiers.
    """
    peak = np.max(np.abs(y))
    if peak > 0:
        return y / peak * target_peak
    return y

#NEEDED 
# ── Pipeline complet pour UN fichier ─────────────────────────────────────────
def process_file(y_raw, sr_raw, file_name, class_name):
    """
    Pipeline unifié (remplace process_one_file + process_audio).
    Retourne (y_processed, sr, meta_dict) ou (None, sr, meta_dict) si rejeté.

    Étapes :
      1. Resample → TARGET_SR
      2. Bandpass filter 100-2000 Hz
      3. Trim silence début ET fin
      4. Rejet si durée < MIN_DURATION_S
      5. Pad / crop → TARGET_DURATION_S
      6. Normalisation amplitude
    """
    original_duration = len(y_raw) / sr_raw

    # 1. Resample
    if sr_raw != TARGET_SR:
        y = librosa.resample(y_raw, orig_sr=sr_raw, target_sr=TARGET_SR)
        sr = TARGET_SR
    else:
        y, sr = y_raw.copy(), sr_raw

    # 2. Bandpass
    y = bandpass_filter(y, sr)

    # 3. Trim silence début ET fin
    y, leading_s, trailing_s = trim_silence(y, sr)
    stripped_duration = len(y) / sr

    # 4. Rejet si trop court
    if stripped_duration < MIN_DURATION_S:
        return None, sr, {
            "FILE_NAME":           file_name,
            "CLASS":               class_name,
            "ACTION":              "SKIPPED_TOO_SHORT",
            "ORIGINAL_DURATION_S": round(original_duration, 4),
            "STRIPPED_DURATION_S": round(stripped_duration, 4),
            "FINAL_DURATION_S":    None,
            "LEADING_SILENCE_S":   round(leading_s, 4),
            "TRAILING_SILENCE_S":  round(trailing_s, 4),
            "AMPLITUDE_MAX":       None,
            "RMS":                 None,
        }

    # 5. Pad / crop
    y = pad_or_crop(y, sr)

    # 6. Normalisation
    y = normalize_amplitude(y)

    # Labelling action
    target_samples = int(TARGET_DURATION_S * sr)
    if leading_s > 0 and trailing_s > 0:
        action = "STRIPPED_BOTH_ENDS"
    elif leading_s > 0:
        action = "STRIPPED_LEADING"
    elif trailing_s > 0:
        action = "STRIPPED_TRAILING"
    elif len(y) == target_samples and original_duration * sr > target_samples:
        action = "CROPPED"
    elif len(y) == target_samples and original_duration * sr < target_samples:
        action = "PADDED"
    else:
        action = "PROCESSED"

    meta = {
        "FILE_NAME":           file_name,
        "CLASS":               class_name,
        "ACTION":              action,
        "ORIGINAL_DURATION_S": round(original_duration, 4),
        "STRIPPED_DURATION_S": round(stripped_duration, 4),
        "FINAL_DURATION_S":    round(len(y) / sr, 4),
        "LEADING_SILENCE_S":   round(leading_s, 4),
        "TRAILING_SILENCE_S":  round(trailing_s, 4),
        "SAMPLE_RATE":         sr,
        "N_SAMPLES":           len(y),
        "AMPLITUDE_MAX":       round(float(np.max(np.abs(y))), 6),
        "RMS":                 round(float(np.sqrt(np.mean(y**2))), 6),
    }
    return y, sr, meta

#NEEDED
# ── Extraction features ───────────────────────────────────────────────────────
def extract_features(y, sr, file_name, class_name, local_feat_dir, feat_metadata_list):
    """Extrait 6 features et enregistre en .npy local."""
    features = {
        "mel":       librosa.power_to_db(
                         librosa.feature.melspectrogram(
                             y=y, sr=sr, n_mels=128, n_fft=2048, hop_length=512),
                         ref=np.max),
        "mfcc":      librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13),
        "chroma":    librosa.feature.chroma_stft(y=y, sr=sr),
        "centroid":  librosa.feature.spectral_centroid(y=y, sr=sr),
        "bandwidth": librosa.feature.spectral_bandwidth(y=y, sr=sr),
        "zcr":       librosa.feature.zero_crossing_rate(y),
    }
    stem = os.path.splitext(file_name)[0]
    for feat_type, feat_data in features.items():
        npy_name = f"{stem}_{feat_type}.npy"
        np.save(os.path.join(local_feat_dir, npy_name), feat_data)
        feat_metadata_list.append({
            "FILE_NAME":    file_name,
            "CLASS":        class_name,
            "FEATURE_TYPE": feat_type,
            "NPY_FILENAME": npy_name,
        })


print("Fonctions audio définies")

In [ ]:
FORBIDDEN_PAIRS = [
    {'pitch_up', 'stretch_fast'},
    {'pitch_down', 'stretch_slow'},
]

def build_transforms(sr):
    return {
        'stretch_slow': lambda y: librosa.effects.time_stretch(y, rate=random.uniform(0.85, 0.95)),
        'stretch_fast': lambda y: librosa.effects.time_stretch(y, rate=random.uniform(1.05, 1.15)),
        'pitch_up':     lambda y: librosa.effects.pitch_shift(y, sr=sr, n_steps=random.uniform(0.5, 2.0)),
        'pitch_down':   lambda y: librosa.effects.pitch_shift(y, sr=sr, n_steps=random.uniform(-2.0, -0.5)),
        'noise':        lambda y: y + random.uniform(0.002, 0.008) * np.random.randn(len(y)),
        'gain':         lambda y: y * random.uniform(0.8, 1.2),
    }

def augment_one(y, sr, variant_id=0):
    """Génère une variante via 2-3 transformations aléatoires enchaînées."""
    transforms = build_transforms(sr)
    n_t = random.choices([2, 3], weights=[0.7, 0.3])[0]
    for _ in range(20):
        chosen = random.sample(list(transforms.keys()), n_t)
        if not any(p.issubset(set(chosen)) for p in FORBIDDEN_PAIRS):
            break
    y_aug = y.copy()
    for name in chosen:
        y_aug = transforms[name](y_aug)
    tag = '+'.join(n[:3] for n in chosen) + f'_v{variant_id}'
    return y_aug, tag


# ── Calcul des cibles d'augmentation ─────────────────────────────────────────
def compute_aug_targets(real_counts, max_ratio=MAX_AUG_RATIO, gen_factor=GENERATION_FACTOR):
    """
    N'augmente QUE les classes sous la médiane.
    Fix v2 : la v1 augmentait toutes les classes y compris COPD.
    """
    counts  = list(real_counts.values())
    median  = np.median(counts)
    targets = {}
    for cls, n_real in real_counts.items():
        if n_real >= median:
            targets[cls] = {'n_real': n_real, 'n_aug_needed': 0, 'n_generate': 0, 'n_total': n_real}
        else:
            n_aug = int(n_real * max_ratio / (1 - max_ratio))
            targets[cls] = {
                'n_real':        n_real,
                'n_aug_needed':  n_aug,
                'n_generate':    n_aug * gen_factor,
                'n_total':       n_real + n_aug,
                'pct_aug':       round(n_aug / (n_real + n_aug) * 100, 1),
            }
    return targets


# ── Scoring MFCC cosinus ─────────────────────────────────────────────────────
def mfcc_vector(y, sr, n_mfcc=N_MFCC_SCORE):
    return np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc), axis=1)

def cosine_score(y, sr, centroid):
    v = mfcc_vector(y, sr).reshape(1, -1)
    return float(cosine_similarity(v, centroid.reshape(1, -1))[0][0])


print("Fonctions augmentation définies")

## 1. Preprocessing des audios réels

In [ ]:
def run_real_preprocessing(session, source_stage, processed_stage, feature_stage,
                           source_meta_table, class_to_dir):
    metadata_df = session.sql(f"SELECT * FROM {source_meta_table}").to_pandas()
    modifications, proc_meta, feat_meta = [], [], []

    with tempfile.TemporaryDirectory() as tmp:
        src_dir  = os.path.join(tmp, "src")
        dst_dir  = os.path.join(tmp, "proc")
        feat_dir = os.path.join(tmp, "feat")

        for class_name, subdir in class_to_dir.items():
            class_files = metadata_df[metadata_df["CLASS"] == class_name]
            if class_files.empty:
                continue

            c_src  = os.path.join(src_dir,  subdir); os.makedirs(c_src,  exist_ok=True)
            c_dst  = os.path.join(dst_dir,  subdir); os.makedirs(c_dst,  exist_ok=True)
            c_feat = os.path.join(feat_dir, subdir); os.makedirs(c_feat, exist_ok=True)

            session.file.get(f"{source_stage}/{subdir}", c_src)
            print(f"[{class_name}] {len(os.listdir(c_src))} fichiers téléchargés")

            for _, row in class_files.iterrows():
                fname      = row["FILE_NAME"]
                local_path = os.path.join(c_src, fname)

                # Décompression .gz si nécessaire
                if not os.path.exists(local_path):
                    gz = local_path + ".gz"
                    if os.path.exists(gz):
                        with gzip.open(gz, 'rb') as fi, open(local_path, 'wb') as fo:
                            shutil.copyfileobj(fi, fo)
                    else:
                        modifications.append({"FILE_NAME": fname, "CLASS": class_name,
                                              "ACTION": "SKIPPED_NOT_FOUND"})
                        continue

                try:
                    y_raw, sr_raw = librosa.load(local_path, sr=None)
                    y, sr, meta  = process_file(y_raw, sr_raw, fname, class_name)
                    modifications.append(meta)

                    if y is not None:
                        out_path = os.path.join(c_dst, fname)
                        sf.write(out_path, y, sr)
                        proc_meta.append(meta)
                        extract_features(y, sr, fname, subdir, c_feat, feat_meta)

                except Exception as e:
                    modifications.append({"FILE_NAME": fname, "CLASS": class_name,
                                          "ACTION": f"ERROR:{str(e)[:120]}"})

            # Upload batch processed + features
            for local_d, stage_p in [(c_dst, f"{processed_stage}/{subdir}"),
                                      (c_feat, f"{feature_stage}/{subdir}")]:
                for f in os.listdir(local_d):
                    try:
                        session.file.put(os.path.join(local_d, f), stage_p,
                                         auto_compress=False, overwrite=True)
                    except Exception as e:
                        print(f"  Upload échoué {f}: {e}")

            n_proc = len([m for m in modifications
                          if m.get('CLASS') == class_name
                          and not str(m.get('ACTION','')).startswith('SKIP')
                          and not str(m.get('ACTION','')).startswith('ERROR')])
            print(f"  → {n_proc} fichiers traités et uploadés")

    return pd.DataFrame(modifications), pd.DataFrame(proc_meta), pd.DataFrame(feat_meta)


mod_df, proc_meta_df, feat_meta_df = run_real_preprocessing(
    session, SOURCE_STAGE, PROCESSED_STAGE, FEATURE_STAGE,
    SOURCE_METADATA_TABLE, CLASS_TO_DIR
)

print(f"\nTotal : {len(mod_df)} | Traités : {(~mod_df['ACTION'].str.startswith('SKIP') & ~mod_df['ACTION'].str.startswith('ERROR')).sum()} | Skipped : {mod_df['ACTION'].str.startswith('SKIP').sum()} | Erreurs : {mod_df['ACTION'].str.startswith('ERROR').sum()}")
print("\nActions :")
print(mod_df['ACTION'].value_counts().to_string())
print("\nPar classe :")
print(mod_df.groupby(['CLASS','ACTION']).size().unstack(fill_value=0).to_string())

In [ ]:
if not proc_meta_df.empty:
    # CREATE OR REPLACE évite le problème de PRIMARY KEY sur re-run
    session.create_dataframe(proc_meta_df).write \
        .mode("overwrite").save_as_table(PROC_METADATA_TABLE)
    print(f"Table créée : {PROC_METADATA_TABLE} ({len(proc_meta_df)} lignes)")

if not feat_meta_df.empty:
    session.create_dataframe(feat_meta_df).write \
        .mode("overwrite").save_as_table(FEATURE_METADATA_TABLE)
    print(f"Table créée : {FEATURE_METADATA_TABLE} ({len(feat_meta_df)} lignes)")

# Distribution réelle après preprocessing (skipped exclus)
real_counts_processed = (
    session.sql(f"""
        SELECT CLASS, COUNT(FILE_NAME) AS N
        FROM {PROC_METADATA_TABLE}
        WHERE ACTION NOT LIKE 'SKIPPED%'
          AND ACTION NOT LIKE 'ERROR%'
        GROUP BY CLASS
    """)
    .to_pandas()
    .set_index('CLASS')['N']
    .to_dict()
)

print("\nDistribution après preprocessing :")
for k,v in sorted(real_counts_processed.items(), key=lambda x: -x[1]):
    print(f"  {k:<12} {v}")

## 2. Génération, preprocessing et sélection des augmentés

1. Générer le signal augmenté brut
2. Passer dans `process_file` (même pipeline que les réels)
3. Scorer par cosine MFCC sur le signal processé
4. Sélectionner les meilleurs

In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

targets = compute_aug_targets(real_counts_processed)

print(f"{'Classe':<12} {'Réels':>6} {'À générer':>10} {'À garder':>9} {'% aug':>7}")
print("-" * 50)
for cls, t in targets.items():
    if t['n_aug_needed'] > 0:
        print(f"  {cls:<12} {t['n_real']:>6} {t['n_generate']:>10} "
              f"{t['n_aug_needed']:>9} {t.get('pct_aug',0):>6}%")
    else:
        print(f"  {cls:<12} {t['n_real']:>6} {'—':>10} {'—':>9} {'0%':>7}")

In [ ]:
def run_augmentation_pipeline(session, source_stage, processed_stage, aug_stage,
                               aug_feat_stage, targets, class_to_dir):
    """
    Pour chaque classe sous-représentée :
    1. Télécharge les réels processés depuis processed_stage
    2. Calcule le centroid MFCC de référence (sur signaux processés)
    3. Génère n_generate candidats → process_file → score cosinus
    4. Sélectionne les n_aug_needed meilleurs
    5. Upload vers aug_stage + aug_feat_stage
    """

    classes_to_aug = {cls: t for cls, t in targets.items() if t['n_aug_needed'] > 0}

    print("Nettoyage des stages augmentés...")
    for cls in classes_to_aug:
        subdir = class_to_dir[cls] 
        for stage in [aug_stage, aug_feat_stage]:
            stage_path = f"{stage}/{subdir}"
            try:
                # LIST first — REMOVE fails silently if path doesn't exist
                files = session.sql(f"LIST {stage_path}").collect()
                if files:
                    session.sql(f"REMOVE {stage_path}").collect()
                    print(f"  Cleared {len(files)} files from {stage_path}")
                else:
                    print(f"  {stage_path} already empty")
            except Exception as e:
                print(f"  Could not clear {stage_path}: {e}")

    aug_meta_all = []
    aug_feat_all = []
    summary      = {}

    with tempfile.TemporaryDirectory() as tmp:
        for cls, t in classes_to_aug.items():
            subdir     = class_to_dir.get(
                next(k for k in class_to_dir if k.lower() == cls.lower()), cls)
            n_needed   = t['n_aug_needed']
            n_generate = t['n_generate']

            print(f"\n{'='*60}")
            print(f"[{cls.upper()}] réels={t['n_real']}  "
                  f"générer={n_generate}  garder={n_needed}")
            print(f"{'='*60}")

            # Dossiers locaux
            c_real = os.path.join(tmp, cls, "real");     os.makedirs(c_real, exist_ok=True)
            c_aug  = os.path.join(tmp, cls, "aug_wav");  os.makedirs(c_aug,  exist_ok=True)
            c_feat = os.path.join(tmp, cls, "aug_feat"); os.makedirs(c_feat, exist_ok=True)

            # Téléchargement des réels processés
            session.file.get(f"{processed_stage}/{subdir}", c_real)
            real_files = [os.path.join(c_real, f) for f in os.listdir(c_real)
                          if f.endswith('.wav')]
            print(f"  {len(real_files)} fichiers réels téléchargés")

            if not real_files:
                print(f"  ERREUR : aucun fichier réel pour {cls}")
                continue

            # Centroid MFCC des réels (déjà processés)
            ref_vectors = []
            for rp in real_files:
                y_r, _ = librosa.load(rp, sr=TARGET_SR)
                ref_vectors.append(mfcc_vector(y_r, TARGET_SR))
            centroid = np.mean(ref_vectors, axis=0)

            # Génération des candidats
            candidates = []  # (score, y_proc, fname_out, meta, feat_meta_list)
            variant_id = 0
            per_file   = max(1, n_generate // len(real_files))

            # Guard : si per_file * n_files < n_generate, compense sur les premiers fichiers
            extra = n_generate - per_file * len(real_files)

            for file_idx, rp in enumerate(real_files):
                y_real, _ = librosa.load(rp, sr=TARGET_SR)
                n_this    = per_file + (1 if file_idx < extra else 0)
                src_stem  = os.path.splitext(os.path.basename(rp))[0]

                for _ in range(n_this):
                    # 1. Augmentation brute
                    y_aug_raw, tag = augment_one(y_real, TARGET_SR, variant_id)
                    variant_id += 1

                    # 2. Process (même pipeline que les réels)
                    y_proc, sr_proc, meta = process_file(
                        y_aug_raw, TARGET_SR,
                        f"{src_stem}_{tag}.wav", cls
                    )

                    # 3. Rejet si trop court après trim
                    if y_proc is None:
                        continue

                    # 4. Score cosinus (sur signal processé)
                    score = cosine_score(y_proc, TARGET_SR, centroid)

                    # Collecte locale des features (calculé si sélectionné)
                    candidates.append((score, y_proc, f"{src_stem}_{tag}.wav", meta))

            print(f"  {len(candidates)} candidats valides scorés")

            if len(candidates) == 0:
                print(f"  ERREUR : aucun candidat valide généré pour {cls}")
                summary[cls] = {'generated': 0, 'selected': 0}
                continue

            # 5. Sélection des meilleurs
            candidates.sort(key=lambda x: x[0], reverse=True)
            selected = candidates[:n_needed]

            scores_sel = [c[0] for c in selected]
            scores_rej = [c[0] for c in candidates[n_needed:]]
            print(f"  Sélectionnés — moy:{np.mean(scores_sel):.4f}  "
                  f"min:{min(scores_sel):.4f}  max:{max(scores_sel):.4f}")
            if scores_rej:
                print(f"  Rejetés      — moy:{np.mean(scores_rej):.4f}  "
                      f"min:{min(scores_rej):.4f}  max:{max(scores_rej):.4f}")

            # 6. Sauvegarde + extraction features des sélectionnés
            for rank, (score, y_proc, fname, meta) in enumerate(selected):
                out_wav  = os.path.join(c_aug,  fname)
                sf.write(out_wav, y_proc, TARGET_SR)

                feat_list = []
                extract_features(y_proc, TARGET_SR, fname, subdir, c_feat, feat_list)

                meta_row = {**meta,
                            "COSINE_SCORE": round(score, 6),
                            "SELECTION_RANK": rank,
                            "IS_AUGMENTED": True}
                aug_meta_all.append(meta_row)
                aug_feat_all.extend(feat_list)

            # Upload batch
            for local_d, stage_p in [(c_aug,  f"{aug_stage}/{subdir}"),
                                      (c_feat, f"{aug_feat_stage}/{subdir}")]:
                for f in os.listdir(local_d):
                    try:
                        session.file.put(os.path.join(local_d, f), stage_p,
                                         auto_compress=False, overwrite=True)
                    except Exception as e:
                        print(f"  Upload échoué {f}: {e}")

            summary[cls] = {'generated': len(candidates), 'selected': len(selected)}
            print(f"  {len(selected)} fichiers uploadés vers {aug_stage}/{subdir}/")

    return pd.DataFrame(aug_meta_all), pd.DataFrame(aug_feat_all), summary


aug_meta_df, aug_feat_meta_df, aug_summary = run_augmentation_pipeline(
    session, PROCESSED_STAGE, PROCESSED_STAGE, AUGMENTED_STAGE,
    AUG_FEATURE_STAGE, targets, CLASS_TO_DIR
)

print("\n--- Résumé augmentation ---")
for cls, s in aug_summary.items():
    print(f"  {cls:<12} générés={s['generated']}  sélectionnés={s['selected']}")

In [ ]:
if not aug_meta_df.empty:
    session.create_dataframe(aug_meta_df).write \
        .mode("overwrite").save_as_table(AUG_METADATA_TABLE)
    print(f"Table créée : {AUG_METADATA_TABLE} ({len(aug_meta_df)} lignes)")

if not aug_feat_meta_df.empty:
    session.create_dataframe(aug_feat_meta_df).write \
        .mode("overwrite").save_as_table(AUG_FEATURE_META_TABLE)
    print(f"Table créée : {AUG_FEATURE_META_TABLE} ({len(aug_feat_meta_df)} lignes)")

## 3. Vue d'entraînement finale

Joint les réels processés et les augmentés sélectionnés.
Ajoute les **class weights** directement dans la vue.

In [ ]:
aug_meta_df = session.sql(f"SELECT * FROM {AUG_METADATA_TABLE}").to_pandas()
aug_feat_meta_df = session.sql(f"SELECT * FROM {AUG_FEATURE_META_TABLE}").to_pandas()

In [ ]:
# Calcul des class weights sur la distribution finale
from sklearn.utils.class_weight import compute_class_weight

class_order = sorted(real_counts_processed.keys())
final_counts = {}
for cls in class_order:
    n_real = real_counts_processed.get(cls, 0)
    n_aug  = len(aug_meta_df[aug_meta_df['CLASS'].str.lower() == cls.lower()]) \
             if not aug_meta_df.empty else 0    
    final_counts[cls] = n_real + n_aug

y_labels = []
for i, cls in enumerate(class_order):
    y_labels.extend([i] * final_counts[cls])

weights = compute_class_weight('balanced', classes=np.unique(y_labels), y=np.array(y_labels))
weight_map = {cls: round(float(weights[i]), 6) for i, cls in enumerate(class_order)}

print("Class weights (distribution finale) :")
for cls, w in sorted(weight_map.items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 8)
    print(f"  {cls:<12} {w:.4f}  {bar}")

# Construction du CASE WHEN pour les weights dans la vue SQL
weight_case = "\n            ".join(
    f"WHEN LOWER(CLASS) = '{cls.lower()}' THEN {w}" for cls, w in weight_map.items()
)

session.sql(f"""
    CREATE OR REPLACE VIEW {TRAINING_VIEW} AS

    WITH real_audio AS (
        SELECT
            FILE_NAME,
            CLASS,
            SAMPLE_RATE,
            FINAL_DURATION_S,
            N_SAMPLES,
            AMPLITUDE_MAX,
            RMS,
            FALSE              AS IS_AUGMENTED,
            NULL               AS COSINE_SCORE,
            NULL               AS SELECTION_RANK,
            CASE {weight_case} ELSE 1.0 END AS CLASS_WEIGHT
        FROM {PROC_METADATA_TABLE}
        WHERE ACTION NOT LIKE 'SKIPPED%'
          AND ACTION NOT LIKE 'ERROR%'
    ),

    aug_audio AS (
        SELECT
            FILE_NAME,
            CLASS,
            SAMPLE_RATE,
            FINAL_DURATION_S,
            N_SAMPLES,
            AMPLITUDE_MAX,
            RMS,
            TRUE               AS IS_AUGMENTED,
            COSINE_SCORE,
            SELECTION_RANK,
            CASE {weight_case} ELSE 1.0 END AS CLASS_WEIGHT
        FROM {AUG_METADATA_TABLE}
    ),

    all_audio AS (
        SELECT * FROM real_audio
        UNION ALL
        SELECT * FROM aug_audio
    ),

    -- Pivot features: one row per file, one column per feature type
    -- NPY_FILENAME gives the stage path to load at training time
    features_wide AS (
        SELECT
            FILE_NAME,
            MAX(CASE WHEN FEATURE_TYPE = 'mel'       THEN NPY_FILENAME END) AS FEAT_MEL,
            MAX(CASE WHEN FEATURE_TYPE = 'mfcc'      THEN NPY_FILENAME END) AS FEAT_MFCC,
            MAX(CASE WHEN FEATURE_TYPE = 'chroma'    THEN NPY_FILENAME END) AS FEAT_CHROMA,
            MAX(CASE WHEN FEATURE_TYPE = 'centroid'  THEN NPY_FILENAME END) AS FEAT_CENTROID,
            MAX(CASE WHEN FEATURE_TYPE = 'bandwidth' THEN NPY_FILENAME END) AS FEAT_BANDWIDTH,
            MAX(CASE WHEN FEATURE_TYPE = 'zcr'       THEN NPY_FILENAME END) AS FEAT_ZCR
        FROM (
            SELECT FILE_NAME, FEATURE_TYPE, NPY_FILENAME FROM {FEATURE_METADATA_TABLE}
            UNION ALL
            SELECT FILE_NAME, FEATURE_TYPE, NPY_FILENAME FROM {AUG_FEATURE_META_TABLE}
        )
        GROUP BY FILE_NAME
    )

    SELECT
        a.*,
        f.FEAT_MEL,
        f.FEAT_MFCC,
        f.FEAT_CHROMA,
        f.FEAT_CENTROID,
        f.FEAT_BANDWIDTH,
        f.FEAT_ZCR
    FROM all_audio a
    LEFT JOIN features_wide f ON a.FILE_NAME = f.FILE_NAME
""").collect()

print(f"\nVue créée : {TRAINING_VIEW}")

## Rapport final

In [ ]:
report = session.sql(f"""
    SELECT
        CLASS,
        COUNT_IF(NOT IS_AUGMENTED)  AS real_samples,
        COUNT_IF(IS_AUGMENTED)      AS augmented_samples,
        COUNT(*)                    AS total,
        ROUND(COUNT_IF(IS_AUGMENTED) / COUNT(*) * 100, 1) AS pct_augmented,
        ROUND(AVG(CLASS_WEIGHT), 4) AS class_weight,
        ROUND(AVG(RMS), 6)          AS avg_rms,
        ROUND(AVG(AMPLITUDE_MAX), 6) AS avg_amplitude_max
    FROM {TRAINING_VIEW}
    GROUP BY CLASS
    ORDER BY total DESC
""").to_pandas()

print(f"\n{'='*80}")
print("TRAINING DATASET — RAPPORT FINAL")
print(f"{'='*80}")
print(report.to_string(index=False))
print(f"\nTotal samples : {report['TOTAL'].sum()}")
print(f"  dont réels   : {report['REAL_SAMPLES'].sum()}")
print(f"  dont augmentés : {report['AUGMENTED_SAMPLES'].sum()}")

# Ratio COPD / Bronchial résiduel
copd_n      = int(report.loc[report['CLASS'].str.lower()=='copd', 'TOTAL'].iloc[0])
bronchial_n = int(report.loc[report['CLASS'].str.lower()=='bronchial', 'TOTAL'].iloc[0])
print(f"\nRatio COPD/Bronchial : {copd_n/bronchial_n:.2f} (était 3.86 avant augmentation)")
print(f"Résiduel couvert par class weights (Bronchial weight = {weight_map.get('bronchial', '?')})")

In [ ]:
SELECT CLASS, IS_AUGMENTED, COUNT(*) AS N
FROM M2_ISD_EQUIPE_1_DB.PROCESSED.TRAINING_DATA_V
GROUP BY CLASS, IS_AUGMENTED
ORDER BY CLASS, IS_AUGMENTED;

In [ ]:
SELECT * 
FROM M2_ISD_EQUIPE_1_DB.PROCESSED.TRAINING_DATA_V
LIMIT 10